# Blindspots — proof of concept

A person says one thing. We show them the tetrad around it, with the corner they are
not looking at — **A+**, the constructive side of the opposing view — in front.

This notebook is the pre-MVP: no app, no server, no UI. It exists to answer two
questions with real output before anyone builds a backend:

1. Is the A+ the framework produces worth showing to a person?
2. Which of the two ways of producing it should the app run?
3. Can a person POLISH their way to the one that lands — react to a card, get a
   different or adjusted one, without it becoming a chat?

**Where it stands (2026-10-02, Sonnet 5, 40 utterances, the pipeline's own coherence
judge and auditors — `docs/dev-notes/antithesis-selection.md`, "the three doors" and
"The one-shot build"):** Path A draws a coherent first tetrad for 25/40 at ~7 s with a
genuine opposing position for 39/40. Path B — the Advisor's `anchor` — used to run a
staged pipeline (headline → antithesis ladder → aspects for a pair never seen whole) that
drew 14/40 coherent at ~55 s with mostly strawman antitheses; since `fac4909` it reasons
the tetrad in ONE call, the same call Path A makes, and then classifies, scores, grounds,
validates and persists it: 21/40 and 21/40 coherent on two generations, 36–38/40 genuine
antitheses, ~43 s. The two paths now draw the same tetrad; Path B adds the graph.
**Since 2026-10-04 (`e3cc495`) the agent checks its own tetrad before the person sees it:**
both paths draw THREE tetrads in parallel and the framework's own coherence check keeps the
one that holds best (`concerns/tetrad_candidates.py`; selection only, never regeneration from
the verdict). Pre-registered, same 40 utterances, fresh judge: Path A 36/40 coherent (from
25/40; 12 improved, 0 worsened) at ~15 s, genuine antitheses 36/40; Path B 30/40 (from 20/40;
13 improved, 3 worsened) at ~34 s, genuine antitheses 36/40. The two runners-up of a build
come back with their verdicts (`SketchTetrad.alternatives`) and can be persisted without a
new reasoning call (`SketchTetrad(sketch=)`) — Path B's polishing uses them below.
So the pre-MVP runs **Path A** for the screen, and Path B is what a "your recurring
blindspots" screen would run, because it is the one with scores and memory.

**Path A** asks the tool-less `Consultant` for the view — one structured turn, no
database. **Path B** keeps the utterance as an `Input` and runs the one-shot build
(`SketchTetrad`: one reasoning call finds the thesis IN the utterance and writes the
tetrad; `IntroducePolarity` and `ExpandPolarity` score, dedup, ground, validate and
commit it).


In [ ]:
from dialectical_framework.dialectical_reasoning import DialecticalReasoning
from dialectical_framework.settings import Settings

# DI setup, once. Reads .env (host obligation 1 — see examples/advisor_chat.py).
container = DialecticalReasoning.setup(Settings.from_env())
print("model:", container.settings().ai_model)

In [ ]:
UTTERANCE = "I should quit my job and start my own company."


## The renderer

Both paths return `graph/views.py::PerspectiveView`. A consumer app never shows
framework terminology, so the view goes through `without_terminology()` — the same
projection the Advisor applies under `hides_terminology` — and the card names the
corners in plain words. `intent` is the reading (the two axes the tetrad was read
along); `complete` says whether all six corners exist.


In [ ]:
from IPython.display import Markdown, display

from dialectical_framework.graph.views import PerspectiveView


def render(view: PerspectiveView, title: str = "") -> None:
    """One card per perspective: A+ first, the two traps after, the poles as context."""
    v = view.without_terminology()
    text = lambda pole: pole.text if pole else "_(not drawn)_"
    lines = [f"### {title}" if title else ""]
    lines += [
        f"**What you might be missing:** {text(v.a_plus)}",
        "",
        f"- Your position: {text(v.t)}",
        f"- What it stands against: {text(v.a)}",
        f"- Your position, developed well: {text(v.t_plus)}",
        "",
        f"- Trap on your side: {text(v.t_minus)}",
        f"- Trap on the other side: {text(v.a_minus)}",
    ]
    if v.intent:
        lines += ["", f"_{v.intent}_"]
    if not v.complete:
        lines += ["", "_(unfinished tetrad — some corners were not drawn)_"]
    display(Markdown("\n".join(lines)))

## Path A — the Consultant's view turn (no graph) — the pre-MVP path

The utterance is the only message in the conversation; `exploration_view(focus=…)` is
three structured, thinking turns in parallel (copies of the same one-message history) that
build the tetrad; the framework's coherence check keeps the best and the record lands in
`messages` once. Nothing is validated and nothing is kept. ~15 s median.


In [ ]:
from dialectical_framework.agents.apps import COUNSELOR_PERSONA
from dialectical_framework.agents.consultant.consultant import Consultant

FOCUS = (
    "the perspective for what I just said: my position, what it stands against, "
    "and above all the constructive side of that opposing view that I am not seeing"
)


async def blindspot_via_consultant(text: str) -> PerspectiveView | None:
    head = Consultant(app_preamble=COUNSELOR_PERSONA, messages=[{"role": "user", "content": text}])
    view = await head.exploration_view(focus=FOCUS)
    return view.perspectives[0] if view.perspectives else None


a = await blindspot_via_consultant(UTTERANCE)
render(a, "Path A — Consultant") if a else print("nothing drawn")

## Polishing — Path A

The promise of the app is not "a tetrad" but the one that lands: *"that's a good one,
I hadn't seen it."* So the first card is a draft and the person's reaction steers the
next. This is not a chat — the person never talks to a counsellor, they react to a card —
and the reactions are a fixed vocabulary of five, each a different recalculation:

| reaction | the person means | what is recalculated |
|---|---|---|
| `accept` | that's it | nothing — the card stands |
| `another` | not my fight / I know this one | a **different tension** from the same material |
| `deeper` | sharper, less safe | the **same tension at higher resolution**, in their own particulars |
| `misread` | that is not my position | the **position re-read** from the material, then its other side |
| `again` | say it differently | the **same tension re-worded**; the model's variance as a feature |

Two meanings, kept apart on purpose: `another`/`accept` are about WHICH tension,
`deeper`/`misread`/`again` about how the same material is read. How a UI elicits these
is the app's business; the framework sees five reasons.

On Path A the Consultant holds the conversation: `exploration_view()` records what it
drew, in words, so the next call with the reaction as `focus` draws AGAINST what was
shown. Nothing is kept beyond `head.messages`.


In [ ]:
class PolishA:
    """One Consultant, many cards. Each reaction becomes the next view's focus."""

    def __init__(self, text: str) -> None:
        self.head = Consultant(app_preamble=COUNSELOR_PERSONA, messages=[{"role": "user", "content": text}])
        self.cards: list[PerspectiveView] = []

    async def first(self) -> list[PerspectiveView]:
        return await self._draw(FOCUS)

    def accept(self) -> PerspectiveView | None:
        """That's it. Nothing recalculated; the last card stands."""
        return self.cards[-1] if self.cards else None

    async def another(self, note: str = "", how_many: int = 1) -> list[PerspectiveView]:
        """Not my fight / I know this one — a DIFFERENT tension, or several."""
        count = {1: "one tension", 2: "two different tensions", 3: "three different tensions"}.get(how_many, "up to four different tensions")
        return await self._draw(
            f"The person set aside what was shown{self._quote(note)}. Draw {count} they have NOT seen "
            "yet — a different tension in what they said, not the shown one re-worded — and above "
            "all the constructive side of the opposing view they are not seeing."
        )

    async def deeper(self, note: str = "") -> list[PerspectiveView]:
        """Sharper: the same tension at higher resolution."""
        return await self._draw(
            f"The person wants the SAME tension as the last card, sharper{self._quote(note)}: the two "
            "constructive sides and the two traps in their own particulars, less safe, more pointed. "
            "Keep the position and what it stands against; raise the resolution."
        )

    async def misread(self, note: str = "") -> list[PerspectiveView]:
        """That is not my position: re-read it from what they said, then its other side."""
        return await self._draw(
            f"The position shown is NOT what the person holds{self._quote(note)}. Re-read their position "
            "from what they said, find what it genuinely stands against, and draw that tetrad."
        )

    async def again(self) -> list[PerspectiveView]:
        """Say it differently: the same tension, fresh wording, same meaning."""
        return await self._draw(
            "Draw the SAME tension as the last card, said differently: fresh wording for every "
            "position, the same meaning, nothing added."
        )

    @staticmethod
    def _quote(note: str) -> str:
        return f' — in their words: "{note}"' if note.strip() else ""

    async def _draw(self, focus: str) -> list[PerspectiveView]:
        view = await self.head.exploration_view(focus=focus)
        start = len(self.cards)
        self.cards.extend(view.perspectives)
        for i, pv in enumerate(view.perspectives, 1):
            render(pv, f"Path A — card {start + i}")
        if not view.perspectives:
            print("nothing drawn")
        return list(view.perspectives)


polish_a = PolishA(UTTERANCE)
await polish_a.first()


In [ ]:
# Not convinced. Two different tensions, then the same one sharper.
await polish_a.another("Too obvious — I know the security argument.", how_many=2)
await polish_a.deeper()


## Path B — the one-shot build into the graph — scores and memory; not the pre-MVP path

What the Advisor's `anchor` runs since `fac4909`, called headlessly inside a Case. The
utterance is NOT the thesis: it is the person's material, kept verbatim as an `Input`
(`AddInput`), and the build's one reasoning call finds the position in it — the thesis
is distilled, not pasted. Then the graph's checks run on what came out: both poles are
classified against the systemic taxonomy, the antithesis scored (HS, Mode, Arousal), the
four aspects scored (complementarity), the tetrad grounded in the utterance's particulars,
validated (coherence + the empirical inequalities), and committed — so a second
utterance from the same person lands next to the first, traces to its own Input, and
`exploration_view()` becomes their map of recurring blindspots.

The host owns the Case (the framework never creates one) and every call runs inside
`scope(sid)`. ~34 s today (three reasoning calls in parallel plus the judge, the rest the
checks, which overlap the opposition's scoring); the `sid:progress` channel exists to show
the person what is happening meanwhile.


In [ ]:
from dialectical_framework.agents.analyst.skills.sketch_tetrad import SketchTetrad
from dialectical_framework.concerns.add_input import capture_input
from dialectical_framework.graph.nodes.case import Case
from dialectical_framework.graph.scope_context import scope
from dialectical_framework.graph.views import exploration_view, perspective_view

# One Case per person. Kept across calls: that is where the memory lives.
case = Case()
case.commit()
print("sid:", case.sid)


async def blindspot_via_pipeline(text: str) -> PerspectiveView | None:
    with scope(case.sid):
        # Whatever the person typed or pasted is MATERIAL, not the thesis: one
        # Input (added and digested), which the build reads and distils the
        # position from. The tetrad traces to it.
        source = await capture_input(text)
        skill = SketchTetrad(input_hashes=[source.hash])
        perspectives = await skill.resolve()
        if not perspectives:
            print("nothing built:", skill.report.summary)
            return None
        return perspective_view(perspectives[0])


b = await blindspot_via_pipeline(UTTERANCE)
render(b, "Path B — one-shot build") if b else None


## Polishing — Path B

The same five reactions, with memory. Every card is a Perspective in the Case; a card
the person moves on from is **set aside with the reaction as its reason** (the graph's
soft-mark — nothing is deleted, and the Advisor would not show it again as new), and the
next build gets the reaction as `context`, the particulars slot every prompt composes
ahead of the material. `deeper` and `again` pin the position so the tension holds;
`another` and `misread` leave it free. The record of reasons is what makes the app an
instrument for the framework: which reaction dominates says which failure dominates.

**Runners-up first.** Every build draws three and keeps one; the other two stay on
`skill.alternatives` with their coherence verdicts. `again` (say it differently) serves a
runner-up with the SAME position, `another` one with a DIFFERENT position — persisted
through `SketchTetrad(sketch=)` with every check but no new reasoning call, so the next
card is there in the time the checks take. When no runner-up fits, a fresh build runs as
before. A served runner-up is not a strawman by construction: it lost only on the
coherence read, by a margin the verdict shows.


In [ ]:
class PolishB:
    """One Case, one Input, many builds. Every reaction is a reason on the graph."""

    def __init__(self, text: str) -> None:
        self.text = text
        self.source = None
        self.cards: list = []        # (Perspective, PerspectiveView)
        self.reasons: list[str] = []  # one per card set aside, in order
        self.spare: list = []        # the last build's runners-up: (sketch, verdict)

    async def first(self):
        with scope(case.sid):
            self.source = await capture_input(self.text)
        return await self._build(context="")

    def accept(self):
        """That's it. The last card stands; nothing is built."""
        return self.cards[-1][1] if self.cards else None

    async def another(self, note: str = ""):
        """Not my fight / I know this one — a DIFFERENT tension."""
        self._set_aside("another", note)
        spare = self._take_spare(same_position=False)
        if spare is not None:
            return await self._persist(spare, f"another{self._quote(note)}")
        return await self._build(
            f"Already shown to the person: {self._shown()}. They set the last one aside{self._quote(note)}. "
            "Build a DIFFERENT tension in the material — not the shown one re-worded."
        )

    async def deeper(self, note: str = ""):
        """Sharper: the same tension at higher resolution; the position stays pinned."""
        last = self._set_aside("deeper", note)
        return await self._build(
            f"The person wants the same tension sharper{self._quote(note)}: the two constructive sides "
            "and the two traps in their own particulars, less safe, more pointed. Keep what the position "
            "stands against.",
            thesis=last.t.text if last and last.t else None,
        )

    async def misread(self, note: str = ""):
        """That is not my position: re-read it from the material, then its other side."""
        self._set_aside("misread", note)
        return await self._build(
            f"The position shown{self._shown(last_only=True)} is NOT what the person holds{self._quote(note)}. "
            "Re-read their position from the material, then find what it genuinely stands against."
        )

    async def again(self):
        """Say it differently: the same tension, fresh wording; the position stays pinned."""
        last = self._set_aside("again", "")
        spare = self._take_spare(same_position=True)
        if spare is not None:
            return await self._persist(spare, "again")
        return await self._build(
            f"The same tension as{self._shown(last_only=True)}, said differently: fresh wording for every "
            "position, the same meaning, nothing added.",
            thesis=last.t.text if last and last.t else None,
        )

    # --- the record ---

    def _set_aside(self, reaction: str, note: str):
        """Mark the last card with the reaction (and the person's words) as its reason."""
        if not self.cards:
            return None
        pp, view = self.cards[-1]
        reason = f"{reaction}: {note}".rstrip(": ")
        with scope(case.sid):
            pp.discarded = reason
            pp.save()
        self.reasons.append(reason)
        return view

    def _shown(self, last_only: bool = False) -> str:
        cards = self.cards[-1:] if last_only else self.cards
        return " " + "; ".join(f"«{pv.t.text} vs {pv.a.text}»" for _pp, pv in cards if pv.t and pv.a)

    @staticmethod
    def _quote(note: str) -> str:
        return f' — in their words: "{note}"' if note.strip() else ""

    # --- the runners-up ---

    def _take_spare(self, same_position: bool):
        """A runner-up whose position is (or is not) the last card's, removed from the spares."""
        last = self.cards[-1][1] if self.cards else None
        shown = (last.t.text if last and last.t else "").strip().lower()
        for i, (sketch, verdict) in enumerate(self.spare):
            if (sketch.thesis.strip().lower() == shown) == same_position:
                del self.spare[i]
                floor = f"{verdict.floor:.2f}" if verdict else "unjudged"
                print(f"(serving a runner-up of the last build — coherence floor {floor}, no new reasoning call)")
                return sketch
        return None

    async def _persist(self, sketch, context: str):
        """A drawn tetrad through every check and into the Case, without re-drawing it."""
        return await self._run(SketchTetrad(input_hashes=[self.source.hash], context=context, sketch=sketch))

    async def _build(self, context: str, thesis: str | None = None):
        return await self._run(SketchTetrad(input_hashes=[self.source.hash], context=context, thesis=thesis))

    async def _run(self, skill):
        with scope(case.sid):
            pps = await skill.resolve()
            if not pps:
                print("nothing built:", skill.report.summary)
                return None
            view = perspective_view(pps[0])
        if skill.alternatives:
            self.spare = list(skill.alternatives)  # a fresh build replaces the spares
        self.cards.append((pps[0], view))
        render(view, f"Path B — card {len(self.cards)}")
        return view


polish_b = PolishB(UTTERANCE)
await polish_b.first()


In [ ]:
# `another` serves a runner-up with a different position when the build left one, else builds.
await polish_b.another("Too obvious — I know the security argument.")
# `again` serves a runner-up with the same position when one is left, else builds.
await polish_b.again()
await polish_b.deeper()


### What Path B knows that Path A cannot say

The same view, before `without_terminology()`: the antithesis's HS (how well the
opposition sits against the taxonomy apex), the complementarity of each corner, the
tetrad's SP, and the validation verdict. None of this exists on Path A — its numbers
are `None` because nothing scored them, not because they are zero.


In [ ]:
if b:
    print("HS(A)        :", b.a.hs)
    print("Ks per corner:", {n: round(p.ks, 2) for n, p in b.poles.items() if p.ks is not None})
    print("metrics      :", b.metrics)
    print("validation   :", b.validation)

### The map after polishing (Path B only)

Two records of the journey. The **view** (`exploration_view()`) shows the standing
cards only — a rejected framing is set aside on the graph, with the person's reason on
the node, and the active view does not show it again as if it were new (the same rule
the Advisor's context follows). The **journey** — every card and every reaction — is
the host's own record, `polish_b.cards` and `polish_b.reasons`, which a "your
blindspots" screen would keep alongside the map.


In [ ]:
with scope(case.sid):
    the_map = exploration_view()
print(f"standing: {len(the_map.perspectives)} card(s); set aside: {sum(1 for pp, _ in polish_b.cards if pp.discarded)}")
for i, p in enumerate(the_map.perspectives, 1):
    render(p, f"standing #{i}")
for (pp, pv), reason in zip(polish_b.cards, polish_b.reasons):
    if pp.discarded:
        print(f"set aside — «{pv.t.text} vs {pv.a.text}» — because: {reason}")


## Where this goes

- **Re-measure before trusting it.** `tests/e2e/probe_view_turn_ask.py` draws the 40
  utterances the way Path A does and audits them; `tests/e2e/probe_tetrad_quality.py`
  runs Path B's build on the same 40 (`TETRAD_PROBE_ATTEMPTS=N` and `VIEW_TURN_ATTEMPTS=N`
  set the draws). Every item is kept as JSON; the best-of-3 figures are one pre-registered
  generation each (`docs/dev-notes/antithesis-selection.md`, "Best-of-N"), not a result.
- **Backend**: a small HTTP service hosting the chosen path — `POST /blindspot {text}` →
  `PerspectiveView.to_dict()`. Host obligations are the documented ones (Case per
  person, `scope(sid)` per request, `principal="human"`, drain at shutdown).
- **Not a tension**: a bare question may draw nothing. The app needs a graceful
  "nothing to draw yet" state; Path A returns an empty view, Path B reports no anchor.
- **Unmeasured**: A+ quality on either path against real users' phrasing. The probe
  is the first sample, not a result.
